# 13B Threshold Pilot

**Purpose.** Small pilot comparing raw vs peak-picked occupancy before the full run.

**Thesis reference.** §5 (pilot)

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:

import json
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

DATA = Path("../ahen_data/filtered_tsv_2")
LABELS = "labels/labels_severity_all.json"
N = 30
IMG = 224
MZ_RANGE, RT_RANGE = (50.0,1650.0), (0.0,37.0)
MZ_FLOOR = 180        # helper drops m/z below this
THRESH, WINDOW = 5, 30

lab = json.load(open(LABELS))["labels"]
def npid(raw):
    d="".join(c for c in raw if c.isdigit()); return f"P{int(d):04d}" if d else None
files=[]
for f in sorted(DATA.glob("*.tsv")):
    p=npid(f.stem.replace("WCmsdata_s",""))
    if p in lab: files.append((f,p))
    if len(files)>=N: break
print(f"pilot on {len(files)} patients\n")

def read(path):
    df=pd.read_csv(path,sep="\t",names=["time","mz","intensity"],dtype=str,low_memory=False)
    for c in ("time","mz","intensity"): df[c]=pd.to_numeric(df[c],errors="coerce")
    return df.dropna()

def pick(df):
    # the helper's df_find_peaks: threshold, then rolling local max
    df=df[df["intensity"]>THRESH].sort_values(["time","mz"])
    keep=df.groupby("time",group_keys=False).apply(
        lambda x: x.loc[x["intensity"]==x["intensity"].rolling(
            WINDOW,center=True,min_periods=1).max()])
    return keep.reset_index(drop=True)

def occ(df):
    m=(df.mz>=MZ_RANGE[0])&(df.mz<MZ_RANGE[1])&(df.time>=RT_RANGE[0])&(df.time<RT_RANGE[1])
    d=df[m]
    H=np.zeros((IMG,IMG))
    r=((d.time-RT_RANGE[0])/(RT_RANGE[1]-RT_RANGE[0])*IMG).astype(int).clip(0,IMG-1)
    c=((d.mz-MZ_RANGE[0])/(MZ_RANGE[1]-MZ_RANGE[0])*IMG).astype(int).clip(0,IMG-1)
    np.add.at(H,(r,c),d.intensity.values)
    return (H>0).mean(), len(d)

print(f"{'patient':9s} {'raw rows':>10s} {'raw occ':>8s} {'picked':>8s} {'pick occ':>9s} {'>180 occ':>9s}")
print("-"*62)
ro,po,fo=[],[],[]
for f,p in files:
    df=read(f)
    o_raw,n_raw=occ(df)
    pk=pick(df)
    o_pick,n_pick=occ(pk)
    o_180,_=occ(pk[pk.mz>=MZ_FLOOR])
    ro.append(o_raw); po.append(o_pick); fo.append(o_180)
    print(f"{p:9s} {n_raw:10d} {o_raw:8.1%} {n_pick:8d} {o_pick:9.1%} {o_180:9.1%}")

print(f"\nmean occupancy   raw {np.mean(ro):.1%}   picked {np.mean(po):.1%}   "
      f"picked+>180 {np.mean(fo):.1%}")
print(f"mean peaks per patient after picking: {np.mean([occ(pick(read(f)))[1] for f,_ in files[:5]]):.0f}")

pilot on 30 patients

patient     raw rows  raw occ   picked  pick occ  >180 occ
--------------------------------------------------------------


/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(


P0001       28126863   100.0%    82662     26.4%     23.3%


/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(


P0002       28146355   100.0%    84165     26.8%     23.6%


/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(


... (54 outputs left out)

P0034       28018051   100.0%    79278     25.7%     22.7%

mean occupancy   raw 100.0%   picked 26.3%   picked+>180 23.1%


/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(
/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is depre

mean peaks per patient after picking: 84854


/tmp/ipykernel_3149773/4075737230.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep=df.groupby("time",group_keys=False).apply(
